# Macro-averaged LoRA fine-tuning for PlurVA (zh/id/si)

Standalone notebook version of `scripts/prepare_training_data.py` + `scripts/train_macro_lora_pt.py`.

Loads the raw PlurVA **dev sets** directly (`data/chinese_dev.jsonl`, `data/indonesian_dev.jsonl`,
`data/sri_lankan_dev.jsonl`), builds the same MCQ prompt/completion pairs used for the zero-shot
baseline eval, splits each language into train/val, then LoRA fine-tunes Qwen3.5-4B on all three
languages simultaneously with an equal-weight (macro-averaged) loss.

Runs on CUDA, MPS, or CPU (auto-detected, CUDA preferred).

In [ ]:
# unsloth isn't in this venv yet -- installs its LoRA/kernel-patch stack.
# `unsloth` must be imported before torch/transformers/peft (next cell) for its
# patches to actually apply; importing it later silently loses the speed/memory wins.
%pip install -q unsloth unsloth_zoo


In [ ]:
from unsloth import FastLanguageModel

import json
import random
import shutil
import time
from collections import Counter
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


## Config

In [ ]:
DATA_DIR = Path("data")
LANG_FILES = {
    "zh": DATA_DIR / "chinese_dev.jsonl",
    "id": DATA_DIR / "indonesian_dev.jsonl",
    "si": DATA_DIR / "sri_lankan_dev.jsonl",
}
LANGS = ["zh", "id", "si"]

MODEL_ID = "Qwen/Qwen3.5-4B"
ADAPTER_PATH = Path("adapters/macro_lora_pt")

VAL_FRACTION = 0.15          # matches prepare_training_data.py
SPLIT_SEED = 42              # matches prepare_training_data.py

MAX_SEQ_LENGTH = 768
PER_LANG_BATCH_SIZE = 2      # examples per language per step; total batch = this x 3
ITERS = 500
STEPS_PER_REPORT = 10
STEPS_PER_EVAL = 5
VAL_BATCHES = 10
PATIENCE = 10                # stop after this many consecutive evals with no macro val-loss improvement (0 disables)
LEARNING_RATE = 1e-4
LORA_RANK = 16
LORA_ALPHA = 32.0
GRAD_CHECKPOINT = True
LOAD_IN_4BIT = False        # LoRA on the full-precision base, matching train_macro_lora_pt.py (no quantization)
TRAIN_SEED = 42


## Data loading & prompt templates

Ported from `scripts/eval_baseline.py` so the training prompt format is identical to what the
zero-shot baseline eval uses (scenario-inclusion, Sinhala `Both`/`0` -> fixed C/D meta-options,
Indonesian majority-vote gold resolution).

In [ ]:
SI_FIXED_OPTION_C = 'පිළිතුරු දෙකම නිවැරදියි.'
SI_FIXED_OPTION_D = 'පිළිතුරු දෙකම නිවැරදි නොවේ.'
SI_GOLD_MAP = {"Both": "C", "0": "D"}

PROMPT_TEMPLATES = {
    "zh": """You are a Simplified Chinese Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
    "id": """You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
    "si": """You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
}


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_rows(lang):
    return read_jsonl(LANG_FILES[lang])


def resolve_gold(lang, gold_answer):
    """Handle single-letter gold answers, Indonesian's comma-separated
    annotator votes (majority vote; drop rows with no strict majority), and
    Sri Lankan's 'Both'/'0' labels (map to the fixed C/D meta-options)."""
    gold_answer = gold_answer.strip()
    if lang == "si" and gold_answer in SI_GOLD_MAP:
        return SI_GOLD_MAP[gold_answer]
    if "," in gold_answer:
        votes = [v.strip() for v in gold_answer.split(",")]
        counts = Counter(votes)
        top, top_n = counts.most_common(1)[0]
        if list(counts.values()).count(top_n) > 1:
            return None
        return top
    return gold_answer


def resolve_options(lang, row):
    """Return {letter: option_text} actually presented in the prompt."""
    options = {
        "A": row.get("Option_A", ""),
        "B": row.get("Option_B", ""),
        "C": row.get("Option_C", ""),
        "D": row.get("Option_D", ""),
    }
    if lang == "si":
        options["C"] = SI_FIXED_OPTION_C
        options["D"] = SI_FIXED_OPTION_D
    return options


def build_prompt(lang, row, options):
    scenario = row.get("Scenario", "").strip()
    scenario_line = f"Scenario: {scenario}\n" if scenario else ""
    return PROMPT_TEMPLATES[lang].format(
        scenario_line=scenario_line,
        question=row["Question"],
        option_a=options["A"],
        option_b=options["B"],
        option_c=options["C"],
        option_d=options["D"],
    )


## Build train/val examples per language

Same procedure as `scripts/prepare_training_data.py`: resolve gold -> build prompt -> shuffle
(fixed seed) -> slice off `VAL_FRACTION` as the held-out validation split. Output schema is
`{"prompt": <raw un-templated prompt>, "completion": " <letter>"}` -- chat-template application
happens later, in the dataset class, not here (so it can match the zero-shot eval's formatting
exactly).

In [ ]:
def build_examples(lang):
    rows = load_rows(lang)
    examples = []
    dropped = 0
    for row in rows:
        gold = resolve_gold(lang, row["Gold_Answer"])
        if gold is None:
            dropped += 1
            continue
        options = resolve_options(lang, row)
        prompt = build_prompt(lang, row, options)
        examples.append({"prompt": prompt, "completion": f" {gold}"})
    return examples, dropped


train_examples = {}
val_examples = {}
split_rng = random.Random(SPLIT_SEED)
for lang in LANGS:
    examples, dropped = build_examples(lang)
    split_rng.shuffle(examples)
    n_val = max(1, int(len(examples) * VAL_FRACTION))
    val_examples[lang] = examples[:n_val]
    train_examples[lang] = examples[n_val:]
    print(f"{lang}: {len(train_examples[lang])} train, {len(val_examples[lang])} val, "
          f"{dropped} dropped (no majority)")


## Tokenization

`PromptCompletionDataset` applies the chat template (with `enable_thinking=False`) to the raw
prompt, tokenizes prompt+completion, and masks the prompt span with `-100` so the loss is only
computed over the completion tokens -- the standard HF convention for "ignore this position".

In [ ]:
class PromptCompletionDataset:
    def __init__(self, rows, tokenizer, max_seq_length):
        self.examples = []
        for row in rows:
            messages = [{"role": "user", "content": row["prompt"]}]
            try:
                chat_prompt = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False,
                    enable_thinking=False,
                )
            except TypeError:
                chat_prompt = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False,
                )
            prompt_ids = tokenizer(chat_prompt, add_special_tokens=False)["input_ids"]
            completion_ids = tokenizer(row["completion"], add_special_tokens=False)["input_ids"]
            input_ids = prompt_ids + completion_ids
            if len(input_ids) > max_seq_length:
                input_ids = input_ids[-max_seq_length:]
                prompt_len = max(0, len(input_ids) - len(completion_ids))
            else:
                prompt_len = len(prompt_ids)
            labels = [-100] * prompt_len + input_ids[prompt_len:]
            self.examples.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def sample_batch(dataset, batch_size, rng):
    """Sample WITH REPLACEMENT so smaller-language datasets can still fill a
    full batch_size slice every step (the "loop"/upsample behavior)."""
    idxs = [rng.randrange(len(dataset)) for _ in range(batch_size)]
    return [dataset[i] for i in idxs]


def collate(examples, pad_token_id, device):
    max_len = max(len(e["input_ids"]) for e in examples)
    input_ids = torch.full((len(examples), max_len), pad_token_id, dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
    for i, e in enumerate(examples):
        L = len(e["input_ids"])
        input_ids[i, :L] = torch.tensor(e["input_ids"], dtype=torch.long)
        labels[i, :L] = torch.tensor(e["labels"], dtype=torch.long)
        attention_mask[i, :L] = 1
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
    }


## Model + LoRA setup

Uses Unsloth's `FastLanguageModel` instead of raw `transformers.AutoModelForCausalLM` + `peft.get_peft_model` (pattern taken from `Qwen3_5_(4B)_Vision_GRPO.ipynb`'s `FastVisionModel` calls, adapted to the text-only `FastLanguageModel` API since this task has no image input). All hyperparameters (`LORA_RANK`, `LORA_ALPHA`, `MAX_SEQ_LENGTH`, target modules, `GRAD_CHECKPOINT`) are unchanged from the config cell -- only the loading/wrapping mechanism changes. The training loop, loss/accuracy functions, and checkpointing below are untouched.

**Caveats:**
- Unsloth requires an NVIDIA GPU -- there is no MPS/CPU fallback like `train_macro_lora_pt.py` had.
- Qwen3.5's Gated DeltaNet hybrid-attention layers are novel enough that mlx-vlm's Metal kernels don't even support their backward pass (see `train_macro_lora_pt.py`'s docstring). Unsloth ships `unsloth/Qwen3.5-4B` support (confirmed by the reference notebook), but whether its fused Triton kernels specifically optimize the Gated DeltaNet layers -- vs. just wrapping them with standard ops while accelerating the surrounding q/k/v/o/mlp projections -- isn't verifiable from this machine (no internet access to check). Functionally it should still train correctly either way, since LoRA only touches the linear projections regardless of the attention mechanism around them; the risk is only that the speedup may be smaller than Unsloth's usual gains. Worth doing a short smoke-test run before committing to a full training run.


In [ ]:
device = "cuda"  # Unsloth is CUDA-only -- see the caveats above
assert torch.cuda.is_available(), "Unsloth requires CUDA; no MPS/CPU fallback is available with this backend."

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect: bf16 on Ampere+, fp16 otherwise (mirrors the plain-PyTorch script's fallback)
    load_in_4bit=LOAD_IN_4BIT,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth" if GRAD_CHECKPOINT else False,
    random_state=TRAIN_SEED,
)
model.print_trainable_parameters()
FastLanguageModel.for_training(model)  # ensure LoRA layers are in training mode, not Unsloth's fast-inference path


In [ ]:
train_datasets = {
    lang: PromptCompletionDataset(train_examples[lang], tokenizer, MAX_SEQ_LENGTH)
    for lang in LANGS
}
val_datasets = {
    lang: PromptCompletionDataset(val_examples[lang], tokenizer, MAX_SEQ_LENGTH)
    for lang in LANGS
}


## Training loop functions

Macro (equal-weight across languages) loss/accuracy, implemented as three separate `backward()`
calls -- one per language, each contributing 1/3 of the gradient -- rather than one combined loss
tensor, so each language's forward computation graph is freed immediately instead of all three
staying alive simultaneously (Qwen3.5's linear-attention fallback is already memory-hungry per
forward pass).

In [ ]:
def _language_batch_loss(model, examples, pad_token_id, device):
    batch = collate(examples, pad_token_id, device)
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out.logits[:, :-1, :]
    labels = batch["labels"][:, 1:]

    token_ce = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view(labels.shape)
    valid = (labels != -100)
    tok_count = valid.sum(dim=1).clamp(min=1)
    per_example_loss = (token_ce * valid).sum(dim=1) / tok_count
    with torch.no_grad():
        correct = (logits.argmax(dim=-1) == labels) & valid
        accuracy = (correct.sum().float() / valid.sum().clamp(min=1)).item()
    return per_example_loss.mean(), accuracy


def macro_lang_backward(model, batches_by_lang, pad_token_id, device):
    total = 0.0
    lang_acc = {}
    for lang in LANGS:
        lang_loss, acc = _language_batch_loss(model, batches_by_lang[lang], pad_token_id, device)
        (lang_loss / len(LANGS)).backward()
        total += lang_loss.item()
        lang_acc[lang] = acc
    return total / len(LANGS), lang_acc


@torch.no_grad()
def per_language_validate(model, val_datasets, batch_size, val_batches, pad_token_id, device, rng):
    """Returns the macro (equal-weight across languages) val loss, mirroring
    the macro-averaged training objective, so early stopping watches the same
    quantity the model is actually optimized for."""
    model.eval()
    lang_losses = {}
    for lang, ds in val_datasets.items():
        losses = []
        for _ in range(val_batches):
            batch = collate(sample_batch(ds, batch_size, rng), pad_token_id, device)
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            logits = out.logits[:, :-1, :]
            labels = batch["labels"][:, 1:]
            token_ce = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                ignore_index=-100, reduction="none",
            ).view(labels.shape)
            valid = (labels != -100)
            tok_count = valid.sum(dim=1).clamp(min=1)
            per_example_loss = (token_ce * valid).sum(dim=1) / tok_count
            losses.append(per_example_loss.mean().item())
        lang_losses[lang] = sum(losses) / len(losses)
        tqdm.write(f"  val_loss[{lang}]={lang_losses[lang]:.4f}")
    model.train()
    return sum(lang_losses.values()) / len(lang_losses)


def save_best_known(model, adapter_path, best_adapter_path):
    """Fixed-interval periodic save mirrors the best-known checkpoint rather
    than the current in-memory weights, so adapter_path is never worse than
    best_adapter_path (which matters once early stopping has fired, since by
    then the live model is `patience` evals past the actual best)."""
    if best_adapter_path.exists():
        shutil.copytree(best_adapter_path, adapter_path, dirs_exist_ok=True)
    else:
        model.save_pretrained(str(adapter_path))


## Run training

Saves the best macro-val-loss adapter to `adapters/macro_lora_pt/best/` whenever it improves
(note: the best checkpoint is only ever written to disk -- it is *not* reloaded back into the
live model, including on early stopping, so the plain `ADAPTER_PATH` at the end holds the last
iteration's weights, not the best one).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
rng = random.Random(TRAIN_SEED)

ADAPTER_PATH.mkdir(parents=True, exist_ok=True)
best_adapter_path = ADAPTER_PATH / "best"
best_val_loss = float("inf")
stale_evals = 0

losses, steps, t0 = 0.0, 0, time.time()
lang_acc_sums = {lang: 0.0 for lang in LANGS}
print("Starting training (simultaneous zh/id/si, macro-averaged loss)...")
pbar = tqdm(range(1, ITERS + 1), desc="train", unit="it")
for it in pbar:
    step_t0 = time.time()
    batches_by_lang = {
        lang: sample_batch(train_datasets[lang], PER_LANG_BATCH_SIZE, rng)
        for lang in LANGS
    }
    optimizer.zero_grad()
    loss_value, lang_acc = macro_lang_backward(model, batches_by_lang, tokenizer.pad_token_id, device)
    optimizer.step()
    step_dt = time.time() - step_t0

    losses += loss_value
    steps += 1
    for lang in LANGS:
        lang_acc_sums[lang] += lang_acc[lang]
    pbar.set_postfix(loss=f"{loss_value:.4f}")

    acc_str = " ".join(f"acc[{lang}]={lang_acc[lang]:.3f}" for lang in LANGS)
    tqdm.write(f"[iter {it}/{ITERS}] loss={loss_value:.4f} {acc_str} dt={step_dt:.1f}s")

    if it % STEPS_PER_REPORT == 0 or it == ITERS:
        elapsed = time.time() - t0
        acc_str = " ".join(f"acc[{lang}]={lang_acc_sums[lang]/steps:.3f}" for lang in LANGS)
        tqdm.write(f"[iter {it}] train_loss(macro)={losses/steps:.4f} {acc_str} elapsed={elapsed:.0f}s")
        losses, steps = 0.0, 0
        lang_acc_sums = {lang: 0.0 for lang in LANGS}

    if it % STEPS_PER_EVAL == 0 or it == ITERS:
        val_loss = per_language_validate(
            model, val_datasets, PER_LANG_BATCH_SIZE, VAL_BATCHES,
            tokenizer.pad_token_id, device, rng,
        )
        tqdm.write(f"[iter {it}] val_loss(macro)={val_loss:.4f}")
        stop_early = False
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            stale_evals = 0
            model.save_pretrained(str(best_adapter_path))
            tqdm.write(f"[iter {it}] New best macro val loss; saved best adapter to {best_adapter_path}")
        else:
            stale_evals += 1
            tqdm.write(f"[iter {it}] No improvement ({stale_evals}/{PATIENCE})")
            if PATIENCE > 0 and stale_evals >= PATIENCE:
                tqdm.write(f"[iter {it}] Early stopping: no macro val loss improvement for {PATIENCE} evals.")
                stop_early = True
        save_best_known(model, ADAPTER_PATH, best_adapter_path)
        tqdm.write(f"[iter {it}] Saved best-known LoRA adapter to {ADAPTER_PATH}")
        if stop_early:
            break

save_best_known(model, ADAPTER_PATH, best_adapter_path)
print(f"Saved final (best-known) LoRA adapter to {ADAPTER_PATH}")
